# Case study: drift, retraining and the final leaderboard submission

Session 16, block 3 · *Practice:* the 2022 labels are released as feedback data; detect the shift (negative share 19 % → 26 %), retrain and make the final leaderboard submission, ranked on the 2023 reviews.

The situation: model **v1** (TF-IDF + logistic regression, trained on reviews up to 2021) has been serving predictions during 2022 and 2023. Its users send reviews, and the service logs them. Labels arrive late: today the lecturer releases the star ratings of **2022** as feedback. The 2023 labels remain hidden; they rank the final submission (leaderboard round L4).

Plan:

1. train v1 (the model of the sentiment service in `../workspace`),
2. **monitoring without labels**: input drift (KS test, PSI) and prediction drift,
3. **feedback arrives**: label shift and the performance of v1 on 2022,
4. the **retraining decision**, candidate models and a fair comparison on the newest labelled data,
5. **versioning**: save v2 with its metadata, and write the submission.

**Data.** `case-study/data/train_sample.parquet`, `test.parquet` and the feedback file `case-study/data/feedback_2022.csv` (columns `review_id,label`) that the lecturer shares.

**Run with:** `uv run --with jupyterlab --with pandas --with pyarrow --with scikit-learn --with scipy --with matplotlib jupyter lab`

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data"
LABELS = ["neg", "neu", "pos"]

train = pd.read_parquet(DATA / "train_sample.parquet")
test = pd.read_parquet(DATA / "test.parquet")
for df in (train, test):
    df["input"] = np.where(df["title"].fillna("") != "", df["title"].fillna("") + ". " + df["text"].fillna(""),
                           df["text"].fillna(""))
    df["year"] = df["date"].dt.year
print(len(train), "training reviews", train["date"].min().date(), "to", train["date"].max().date())
print(test["year"].value_counts().sort_index().to_dict())          # {2022: 40784, 2023: 10652}

## 1. Model v1

The same pipeline as `workspace/src/sentiment_service/model.py`. Validation is time-based: the newest 20 % of the training reviews are held out.

In [ ]:
def build_pipeline():
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=200_000),
                         LogisticRegression(C=4, class_weight="balanced", max_iter=2000))

train = train.sort_values("date").reset_index(drop=True)
cut = int(len(train) * 0.8)
v1 = build_pipeline().fit(train["input"][:cut], train["label"][:cut])
f1_validation = f1_score(train["label"][cut:], v1.predict(train["input"][cut:]), average="macro")
print(f"v1 validation macro-F1 (newest 20 % up to 2021): {f1_validation:.3f}")   # about 0.69
v1 = build_pipeline().fit(train["input"], train["label"])         # deployed model: all training data

## 2. Monitoring without labels

### 2a. Input drift: text length

The inputs are available at once, labels are not. We compare one numeric feature, the log of the text length, between the **reference** (training data) and the **current** data (2022, 2023).

- The **Kolmogorov–Smirnov (KS) statistic** D is the largest vertical distance between the two empirical distribution functions; it lies between 0 and 1. With tens of thousands of reviews, even a tiny D is "significant", so we read D as an effect size.
- The **population stability index (PSI)** bins the reference into deciles, compares the shares p (reference) and q (current) per bin and sums (q − p) · ln(q / p). Rule of thumb: below 0.1 stable, 0.1–0.25 moderate shift, above 0.25 large shift.

In [ ]:
def psi(reference, current, bins=10, eps=1e-4):
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1))[1:-1])
    p = np.bincount(np.searchsorted(edges, reference, side="right"), minlength=len(edges) + 1) / len(reference)
    q = np.bincount(np.searchsorted(edges, current, side="right"), minlength=len(edges) + 1) / len(current)
    p, q = np.clip(p, eps, None), np.clip(q, eps, None)
    return float(np.sum((q - p) * np.log(q / p)))

ref_len = np.log1p(train["input"].str.len())
rows = []
for year, part in test.groupby("year"):
    cur_len = np.log1p(part["input"].str.len())
    ks = stats.ks_2samp(ref_len, cur_len)
    rows.append({"period": year, "median_chars": part["input"].str.len().median(),
                 "KS_D": round(ks.statistic, 3), "KS_p": f"{ks.pvalue:.1e}", "PSI": round(psi(ref_len, cur_len), 3)})
pd.DataFrame([{"period": "train (to 2021)", "median_chars": train["input"].str.len().median()}] + rows)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.linspace(0, 9, 46)
ax.hist(ref_len, bins=bins, density=True, histtype="step", lw=2, label="train (to 2021)")
ax.hist(np.log1p(test.loc[test["year"] == 2022, "input"].str.len()), bins=bins, density=True,
        histtype="step", lw=2, label="2022")
ax.set(xlabel="log(1 + characters)", ylabel="density", title="Input drift: review length")
ax.legend(frameon=False)
plt.show()

### 2b. Other inputs and prediction drift

Prediction drift compares the distribution of the model's *outputs*. It needs no labels either: if the predicted negative share rises, either the world changed or the model reacts to something else (for example longer texts).

In [ ]:
pred = {"train (to 2021)": v1.predict(train["input"][cut:]),             # held-out reviews of the training period
        2022: v1.predict(test.loc[test["year"] == 2022, "input"]),
        2023: v1.predict(test.loc[test["year"] == 2023, "input"])}
shares = pd.DataFrame({k: pd.Series(v).value_counts(normalize=True).reindex(LABELS) for k, v in pred.items()}).round(3)
print("predicted label shares:")
print(shares)
counts = np.array([pd.Series(v).value_counts().reindex(LABELS).to_numpy() for v in pred.values()])
print("chi-square test, same predicted mix in all periods: p =", f"{stats.chi2_contingency(counts).pvalue:.1e}")
print("verified-purchase share:", round(train["verified_purchase"].mean(), 3), "->",
      test.groupby("year")["verified_purchase"].mean().round(3).to_dict())

**Interim conclusion (before labels arrive).** Reviews became longer (moderate PSI) and the model predicts clearly more negative reviews in 2022–2023. Both are warnings, not proof that the model got worse. We need labels.

## 3. The feedback data: label shift and the performance of v1

The lecturer releases the labels of 2022 (`feedback_2022.csv`). The next cell reads it; on the lecturer's machine it can also be built from the hidden solution.

In [ ]:
FEEDBACK = DATA / "feedback_2022.csv"
SOLUTION = DATA / "instructor" / "solution.csv"
if FEEDBACK.exists():
    feedback = pd.read_csv(FEEDBACK)
elif SOLUTION.exists():                                          # lecturer only: same content as the released file
    feedback = pd.read_csv(SOLUTION).query("Usage == 'Public'")[["review_id", "label"]]
    print("feedback built from the instructor solution (Public = 2022)")
else:
    raise FileNotFoundError("Ask the lecturer for case-study/data/feedback_2022.csv")

labelled = test.merge(feedback, on="review_id")                   # the 2022 reviews with their labels
print(len(labelled), "labelled reviews of", sorted(labelled["year"].unique()))

### 3a. Label shift

**Label shift** (also *prior shift*) is a change in the true class shares P(y). It can only be measured once labels arrive.

In [ ]:
true_shares = pd.DataFrame({"train (to 2021)": train["label"].value_counts(normalize=True),
                            "2022 (feedback)": labelled["label"].value_counts(normalize=True)}).reindex(LABELS)
print(true_shares.round(3))                                        # neg 0.192 -> 0.265
p, q = true_shares.iloc[:, 0].to_numpy(), true_shares.iloc[:, 1].to_numpy()
print("PSI of the label shares:", round(float(np.sum((q - p) * np.log(q / p))), 3))
counts = np.array([train["label"].value_counts().reindex(LABELS), labelled["label"].value_counts().reindex(LABELS)])
print("chi-square p-value:", f"{stats.chi2_contingency(counts).pvalue:.1e}")

The negative share rose from 19 % to 26 %. With 40,000 reviews this is far beyond chance. The PSI of the shares is small (about 0.03) because PSI was designed for many bins; for three classes, report the shares themselves.

### 3b. Performance of v1 on 2022

In [ ]:
pred_2022 = v1.predict(labelled["input"])
f1_2022 = f1_score(labelled["label"], pred_2022, average="macro")
print(f"macro-F1: validation (to 2021) {f1_validation:.3f} -> 2022 {f1_2022:.3f}")
print(classification_report(labelled["label"], pred_2022, digits=3))
print(pd.DataFrame(confusion_matrix(labelled["label"], pred_2022, labels=LABELS), index=LABELS, columns=LABELS))

Is it only **label shift** (more negative reviews, the same words mean the same) or also **concept drift** (P(y | x) changed, for example new products, new complaints)? One check: per-class recall. Under pure label shift, per-class recall stays about the same; precision of the growing class rises and of the shrinking classes falls.

In [ ]:
recall = lambda y, yhat: pd.Series({c: np.mean(yhat[np.asarray(y) == c] == c) for c in LABELS})
y_val, yhat_val = train["label"][cut:].to_numpy(), build_pipeline().fit(train["input"][:cut], train["label"][:cut]).predict(train["input"][cut:])
pd.DataFrame({"recall, validation": recall(y_val, yhat_val), "recall, 2022": recall(labelled["label"].to_numpy(), pred_2022)}).round(3)

## 4. The retraining decision

A written rule, as in `workspace/src/sentiment_service/drift.py`: retrain when macro-F1 on recently labelled data falls more than 0.03 below the validation score, or when an input feature shifts strongly (PSI > 0.25).

In [ ]:
def retrain_needed(f1_recent, f1_reference, psi_input, tolerance=0.03, psi_threshold=0.25):
    reasons = []
    if f1_recent < f1_reference - tolerance:
        reasons.append(f"macro-F1 {f1_recent:.3f} < {f1_reference:.3f} - {tolerance}")
    if psi_input > psi_threshold:
        reasons.append(f"input PSI {psi_input:.3f} > {psi_threshold}")
    return bool(reasons), reasons

psi_2022 = psi(ref_len, np.log1p(labelled["input"].str.len()))
print(retrain_needed(f1_2022, f1_validation, psi_2022))

**What the numbers say (with the course data).** The rule does not fire: macro-F1 fell only from about 0.69 to 0.69, and per-class recall barely changed. The shift is mainly **label shift**: there are more negative reviews, but the model recognises them as well as before (balanced class weights make it less dependent on the training shares). The prediction drift of section 2 was therefore a real change in the world, not a broken model.

Whatever the rule says, the new labels are valuable training data. We compare candidates **fairly**: they are trained without the newest labelled months and evaluated on them, as the 2023 reviews will be.

- **Train** for the comparison: reviews up to 2021 plus feedback from January to September 2022.
- **Evaluate**: feedback from October to December 2022 (the newest labelled data).

Candidates:

| Model | Training data |
|---|---|
| v1 | up to 2021 (unchanged) |
| v2a | up to 2021 + 2022 feedback |
| v2b | 2019–2022 only (a recent window) |
| v2c | up to 2021 + 2022 feedback, with 2022 reviews weighted 3× |

In [ ]:
late = labelled["date"] >= "2022-10-01"
fb_early, fb_late = labelled[~late], labelled[late]
print(len(fb_early), "feedback reviews for training,", len(fb_late), "for evaluation (Oct-Dec 2022)")

def fit(frame, weights=None):
    pipe = build_pipeline()
    return pipe.fit(frame["input"], frame["label"], logisticregression__sample_weight=weights)

base = train[["input", "label", "year"]]
early = fb_early[["input", "label", "year"]]
candidates = {
    "v1": lambda: v1,
    "v2a": lambda: fit(pd.concat([base, early])),
    "v2b": lambda: fit(pd.concat([base[base["year"] >= 2019], early])),
    "v2c": lambda: fit(pd.concat([base, early]), np.r_[np.ones(len(base)), np.full(len(early), 3.0)]),
}
results, fitted = [], {}
for name, make in candidates.items():
    fitted[name] = make()
    yhat = fitted[name].predict(fb_late["input"])
    per_class = f1_score(fb_late["label"], yhat, average=None, labels=LABELS)
    results.append({"model": name, "macro_F1": f1_score(fb_late["label"], yhat, average="macro"),
                    **{f"F1_{c}": v for c, v in zip(LABELS, per_class)},
                    "pred_neg_share": np.mean(yhat == "neg")})
comparison = pd.DataFrame(results).set_index("model").round(3)
print("true negative share Oct-Dec 2022:", round(np.mean(fb_late["label"] == "neg"), 3))
comparison

Read the table: which candidate has the highest macro-F1 on the newest labelled months, and by how much? A difference of a few thousandths on 5,000 reviews may be noise; a bootstrap of the evaluation set (Session 7) gives an interval.

In [ ]:
rng = np.random.default_rng(0)
best = comparison["macro_F1"].idxmax()
y_late = fb_late["label"].to_numpy()
pred_best, pred_v1 = fitted[best].predict(fb_late["input"]), fitted["v1"].predict(fb_late["input"])
diffs = []
for _ in range(200):
    idx = rng.integers(0, len(y_late), len(y_late))
    diffs.append(f1_score(y_late[idx], pred_best[idx], average="macro") - f1_score(y_late[idx], pred_v1[idx], average="macro"))
print(f"{best} minus v1: {np.mean(diffs):.3f}, 95 % interval [{np.quantile(diffs, 0.025):.3f}, {np.quantile(diffs, 0.975):.3f}]")

## 5. Versioning and the final submission

The chosen recipe is refitted on **all** labelled data (up to 2021 and all of 2022) and saved as **v2** with its metadata: version, training data, validation scheme and scores, library version and the reference statistics that monitoring of v2 will use. The old version stays available for a rollback.

In [ ]:
import joblib

recipe = best if best != "v1" else "v2a"
full = pd.concat([base, labelled[["input", "label", "year"]]])
weights = np.r_[np.ones(len(base)), np.full(len(labelled), 3.0)] if recipe == "v2c" else None
if recipe == "v2b":
    full = full[full["year"] >= 2019]
v2 = fit(full, weights)

out_dir = Path.cwd() / "models_v2"
out_dir.mkdir(exist_ok=True)
joblib.dump(v2, out_dir / "model.joblib")
metadata = {"model_version": "2.0.0", "recipe": recipe, "previous_version": "1.0.0",
            "training_data": {"files": ["train_sample.parquet", "feedback_2022.csv"], "n": len(full),
                              "years": [int(full["year"].min()), int(full["year"].max())]},
            "validation": {"scheme": "trained without Oct-Dec 2022, evaluated on it",
                           "macro_f1": float(comparison.loc[best, "macro_F1"])},
            "scikit_learn": sklearn.__version__,
            "reference": {"label_shares": full["label"].value_counts(normalize=True).round(4).to_dict()}}
(out_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=1)[:600])

In [ ]:
submission = pd.DataFrame({"review_id": test["review_id"], "label": v2.predict(test["input"])})
assert len(submission) == len(test) and submission["label"].isin(LABELS).all()
submission.to_csv(Path.cwd() / "submission_L4.csv", index=False)
print(submission["label"].value_counts(normalize=True).round(3).to_dict())
print("written: submission_L4.csv; upload it to the leaderboard (round L4) with a one-page description")

**Lecturer only.** With the hidden solution on the lecturer's machine, the next cell scores v1 and v2 on both leaderboard parts. Students see the private (2023) score in Session 18.

In [ ]:
if SOLUTION.exists():
    solution = pd.read_csv(SOLUTION).merge(test[["review_id", "input"]], on="review_id")
    for name, model in [("v1", v1), ("v2", v2)]:
        yhat = model.predict(solution["input"])
        scores = {usage: round(f1_score(part["label"], yhat[part.index], average="macro"), 3)
                  for usage, part in solution.groupby("Usage")}
        print(name, scores)       # note: v2 has seen the 2022 labels, so its Public score is not a test score
else:
    print("no solution file: scores are announced in Session 18")

## 6. Exercises

1. **Interpret the drift.** Write three sentences for a product manager: what changed between 2021 and 2022, how you know, and what it means for the model.
2. **Concept drift?** Compare the per-class recall of v1 between validation and 2022. Is the drop in macro-F1 explained by label shift alone?
3. **Retraining recipe.** Add a candidate that uses the full training set (`train.parquet`, 434,000 reviews) plus the 2022 feedback. Is the gain worth the training time?
4. **Model card.** Fill in `../workspace/MODEL_CARD.md` for v2: training data, evaluation on Oct–Dec 2022 with the bootstrap interval, the label shift you found, and when the model should be retrained next.
5. **Monitoring plan.** Which statistics would you compute every week in production, and which thresholds trigger a retraining or a human review?

In [ ]:
# your work here